# Receipt key information extraction + safe auto-accept

**Question:** how many receipts can a model process with no human check, while keeping the error rate on the accepted ones low?

1. Fine-tune **LayoutLMv3** (text + layout + image) for key information extraction on **CORD-v2** (receipts: item name, price, total, …).
2. Compare with a **text-only** baseline (XLM-RoBERTa) to measure what layout adds.
3. Evaluate honestly: entity-level F1 with bootstrap confidence intervals, per-field errors.
4. **Auto-accept:** pick a confidence threshold on the validation set so that accepted receipts are almost always fully correct, then check it on the test set.

Runtime: Colab GPU (T4), about 30 minutes in total.

In [ ]:
!pip -q install -U transformers datasets seqeval accelerate

In [ ]:
import json, random, numpy as np, torch
from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report
from seqeval.metrics.sequence_labeling import get_entities

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Tesla T4


## 1. Data: CORD-v2 → words, boxes, BIO labels

In [ ]:
raw = load_dataset("naver-clova-ix/cord-v2")
print(raw)

def parse(ex):
    gt = json.loads(ex["ground_truth"])
    W, H = ex["image"].size
    words, boxes, tags = [], [], []
    for line in gt["valid_line"]:
        cat = line["category"]
        ws = [w for w in line["words"] if w["text"].strip()]
        for i, w in enumerate(ws):
            q = w["quad"]
            x0, x1 = min(q["x1"], q["x4"]), max(q["x2"], q["x3"])
            y0, y1 = min(q["y1"], q["y2"]), max(q["y3"], q["y4"])
            box = [int(1000 * x0 / W), int(1000 * y0 / H), int(1000 * x1 / W), int(1000 * y1 / H)]
            boxes.append([min(max(v, 0), 1000) for v in box])
            words.append(w["text"])
            tags.append(("B-" if i == 0 else "I-") + cat)
    return {"words": words, "boxes": boxes, "tags": tags}

data = {s: [dict(parse(ex), image=ex["image"].convert("RGB")) for ex in raw[s]] for s in ["train", "validation", "test"]}
labels = sorted({t for s in data for d in data[s] for t in d["tags"]})
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print({s: len(v) for s, v in data.items()}, "| labels:", len(labels))
print(list(zip(data["train"][0]["words"], data["train"][0]["tags"]))[:12])

{'train': 800, 'validation': 100, 'test': 100} | labels: 54
[('1', 'B-menu.cnt'), ('x', 'I-menu.cnt'), ('Nasi', 'B-menu.nm'), ('Campur', 'I-menu.nm'), ('Bali', 'I-menu.nm'), ('75,000', 'B-menu.price'), ('1', 'B-menu.cnt'), ('x', 'I-menu.cnt'), ('Bbk', 'B-menu.nm'), ('Bengil', 'I-menu.nm'), ('Nasi', 'I-menu.nm'), ('125,000', 'B-menu.price')]


## 2. Shared evaluation

- **Entity F1** (seqeval): a field counts as correct only if its type and all its words are right.
- **Bootstrap 95% CI**: resample receipts 1,000 times. With only 100 test receipts, a single number hides a lot of uncertainty.
- **Receipt-level correctness**: a receipt is correct only if *every* field is right — this is what matters for auto-accept.

In [ ]:
def bootstrap_f1(y_true, y_pred, n=1000):
    idx = np.arange(len(y_true)); scores = []
    rng = np.random.default_rng(SEED)
    for _ in range(n):
        s = rng.choice(idx, len(idx))
        scores.append(f1_score([y_true[i] for i in s], [y_pred[i] for i in s]))
    return np.percentile(scores, [2.5, 97.5])

def doc_correct(t, p):
    return set(get_entities(t)) == set(get_entities(p))

def report(name, y_true, y_pred):
    f1 = f1_score(y_true, y_pred); lo, hi = bootstrap_f1(y_true, y_pred)
    acc = np.mean([doc_correct(t, p) for t, p in zip(y_true, y_pred)])
    print(f"{name}: entity F1 = {f1:.4f}  (95% CI {lo:.4f}–{hi:.4f}) | fully correct receipts = {acc:.1%}")
    return f1

## 3. Training both models

Training lives in [`kie.py`](kie.py) so it can run as one long job (about 1 hour on a T4):

- **XLM-RoBERTa** (text only): same words and labels, no coordinates and no image. Only the first sub-token of each word is labeled; the rest get -100. 15 epochs, lr 5e-5, batch 8.
- **LayoutLMv3** (text + layout + image): words come from the dataset's own OCR (`apply_ocr=False`), boxes normalized to 0–1000. 20 epochs, lr 3e-5, batch 4.

For both, the checkpoint with the best **validation** F1 is kept; the test set is used only once, at the end. Validation and test predictions with per-word confidences are saved to `results.pkl`.

In [ ]:
!python kie.py

{'train': 800, 'validation': 100, 'test': 100} labels: 54
...
DONE xlmr
...
DONE layoutlmv3


In [ ]:
import pickle
results = pickle.load(open("results.pkl", "rb"))
base, lm = results["xlmr"], results["layoutlmv3"]
report("XLM-R (text only), test", base["test"][0], base["test"][1])
report("LayoutLMv3,        test", lm["test"][0], lm["test"][1])

XLM-R (text only), test: entity F1 = 0.9326  (95% CI 0.9072–0.9536) | fully correct receipts = 66.0%
LayoutLMv3,        test: entity F1 = 0.9516  (95% CI 0.9285–0.9696) | fully correct receipts = 76.0%


In [ ]:
# Paired bootstrap: is LayoutLMv3 really better, or could the gap be noise?
# Same resampled receipts for both models, so receipt difficulty cancels out.
def paired_bootstrap(T, Pa, Pb, n=1000):
    rng = np.random.default_rng(SEED); idx = np.arange(len(T)); diffs = []
    for _ in range(n):
        s = rng.choice(idx, len(idx))
        t = [T[i] for i in s]
        diffs.append(f1_score(t, [Pb[i] for i in s]) - f1_score(t, [Pa[i] for i in s]))
    diffs = np.array(diffs)
    return diffs.mean(), np.percentile(diffs, [2.5, 97.5]), (diffs <= 0).mean()

d, (lo, hi), p = paired_bootstrap(lm["test"][0], base["test"][1], lm["test"][1])
print(f"F1 gain from layout: {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f}); share of resamples with no gain: {p:.3f}")
ra = np.array([doc_correct(t, q) for t, q in zip(base["test"][0], base["test"][1])])
rb = np.array([doc_correct(t, q) for t, q in zip(lm["test"][0], lm["test"][1])])
print(f"Receipts fixed by layout: {(~ra & rb).sum()}, broken by layout: {(ra & ~rb).sum()}")

F1 gain from layout: +0.0192 (95% CI +0.0059 to +0.0338); share of resamples with no gain: 0.002
Receipts fixed by layout: 13, broken by layout: 3


## 5. Where does each model fail? (per field, test set)

In [ ]:
print(classification_report(lm["test"][0], lm["test"][1], digits=3))

                          precision    recall  f1-score   support

                menu.cnt      0.986     0.991     0.989       220
      menu.discountprice      0.500     0.600     0.545        10
                menu.etc      0.000     0.000     0.000         3
       menu.itemsubtotal      0.000     0.000     0.000         6
                 menu.nm      0.948     0.952     0.950       251
                menu.num      1.000     0.727     0.842        11
              menu.price      0.960     0.984     0.972       246
            menu.sub.cnt      0.895     1.000     0.944        17
             menu.sub.nm      0.865     0.889     0.877        36
          menu.sub.price      1.000     0.850     0.919        20
          menu.unitprice      0.928     0.955     0.941        67
sub_total.discount_price      1.000     0.857     0.923         7
           sub_total.etc      0.846     0.846     0.846        13
 sub_total.service_price      1.000     1.000     1.000        12
sub_total

In [ ]:
# Fields where layout helps most: F1 per entity type, text-only vs LayoutLMv3
def per_type(T, P):
    rep = classification_report(T, P, output_dict=True, zero_division=0)
    return {k: (v["f1-score"], int(v["support"])) for k, v in rep.items() if not k.endswith("avg")}
a, b = per_type(*base["test"][:2]), per_type(*lm["test"][:2])
rows = sorted(((t, a.get(t, (0, 0))[0], b[t][0], b[t][1]) for t in b), key=lambda r: r[2] - r[1], reverse=True)
print(f"{'field':28s} {'text-only':>9s} {'layout':>7s} {'support':>7s}")
for t, fa, fb, n in rows: print(f"{t:28s} {fa:9.3f} {fb:7.3f} {n:7d}")

field                        text-only  layout support
menu.unitprice                   0.847   0.941      67
sub_total.service_price          0.917   1.000      12
total.total_etc                  0.250   0.333       3
menu.sub.nm                      0.800   0.877      36
total.menuqty_cnt                0.900   0.967      29
total.menutype_cnt               0.769   0.833       7
menu.sub.cnt                     0.882   0.944      17
menu.price                       0.940   0.972     246
sub_total.etc                    0.815   0.846      13
menu.cnt                         0.973   0.989     220
total.changeprice                0.973   0.982      56
total.total_price                0.974   0.979      96
menu.etc                         0.000   0.000       3
menu.itemsubtotal                0.000   0.000       6
menu.num                         0.842   0.842      11
sub_total.discount_price         0.923   0.923       7
sub_total.subtotal_price         0.970   0.970      66
sub_total.

## 6. Safe auto-accept

Receipt confidence = the **lowest** word confidence on the receipt (one uncertain word is enough to send it to a human).
The threshold is chosen on **validation**, then applied unchanged to **test** — choosing it on test would overstate the result.

In [ ]:
def accept_curve(T, P, C):
    conf = np.array([c.min() if len(c) else 0 for c in C])
    ok = np.array([doc_correct(t, p) for t, p in zip(T, P)])
    return conf, ok

def pick_threshold(conf, ok, max_err=0.02):
    best = 1.01
    for t in sorted(set(conf)):
        acc = conf >= t
        if acc.sum() and (~ok[acc]).mean() <= max_err: best = min(best, t)
    return best

vc, vok = accept_curve(*lm["val"]); tc, tok_ = accept_curve(*lm["test"])
for max_err in [0.0, 0.02, 0.05]:
    th = pick_threshold(vc, vok, max_err)
    acc = tc >= th
    print(f"target ≤{max_err:.0%} errors (val) → threshold {th:.4f} | TEST: auto-accepted {acc.mean():.0%}, "
          f"errors among accepted {(~tok_[acc]).sum()}/{acc.sum()}")
print(f"\nWithout a threshold: {tok_.mean():.0%} of test receipts are fully correct.")

target ≤0% errors (val) → threshold 0.9931 | TEST: auto-accepted 50%, errors among accepted 2/50
target ≤2% errors (val) → threshold 0.9873 | TEST: auto-accepted 65%, errors among accepted 5/65
target ≤5% errors (val) → threshold 0.9795 | TEST: auto-accepted 74%, errors among accepted 6/74

Without a threshold: 76% of test receipts are fully correct.


In [ ]:
import matplotlib.pyplot as plt
order = np.argsort(-tc); acc_frac = np.arange(1, len(tc) + 1) / len(tc)
err = np.cumsum(~tok_[order]) / np.arange(1, len(tc) + 1)
plt.figure(figsize=(6, 4)); plt.plot(acc_frac * 100, err * 100)
plt.xlabel("% of receipts auto-accepted (most confident first)"); plt.ylabel("% of accepted with an error")
plt.title("Coverage vs. risk (test)"); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("coverage_risk.png", dpi=150); plt.show()

## 7. Summary

| Model | Entity F1 (test) | 95% CI | Fully correct receipts |
|---|---|---|---|
| XLM-R, text only | 0.933 | 0.907–0.954 | 66% |
| LayoutLMv3 | **0.952** | 0.929–0.970 | **76%** |

- **Layout helps, and it is not noise:** paired bootstrap gain +0.019 F1 (95% CI +0.006 to +0.034); 13 receipts fixed, 3 broken.
- **Layout helps most on fields defined by their position in a row:** unit price (0.85 → 0.94), service charge, sub-items, total quantity.
- **Confidence alone is not a safe auto-accept rule:** a threshold chosen on 100 validation receipts for ≤2% errors gave 5 errors in 65 accepted test receipts (7.7%). With this little validation data the threshold is unstable, and the model is over-confident on some receipts.
- **Next step:** add an independent check that does not rely on the model's confidence — arithmetic consistency (item prices + tax + service = total), the same idea as check digits in a passport MRZ.